In [2]:
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")

df = pd.read_csv("../data/oasis_longitudinal.csv")
df["label"] = (df["CDR"] > 0).astype(int)
df["Sex"] = (df["M/F"] == "M").astype(int)

y = df["label"]
groups = df["Subject ID"]
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

feature_sets = {
    "B (no MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV"],
    "A (with MMSE)": ["Age", "Sex", "EDUC", "SES", "eTIV", "nWBV", "MMSE"],
}

fill = lambda: SimpleImputer(strategy="median")
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Logistic Regression": make_pipeline(fill(), StandardScaler(),
                                         LogisticRegression(max_iter=1000)),
    "Random Forest": make_pipeline(fill(),
                                   RandomForestClassifier(n_estimators=200, max_depth=4, random_state=0)),
    "XGBoost": make_pipeline(fill(),
                             XGBClassifier(n_estimators=100, max_depth=2, learning_rate=0.1,
                                           eval_metric="logloss", random_state=0)),
}

rows = []
for exp_name, feats in feature_sets.items():
    X = df[feats]
    for model_name, model in models.items():
        all_scores = {m: [] for m in scoring}
        for seed in range(10):  # 10 different random groupings of people
            cv = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
            res = cross_validate(model, X, y, groups=groups, cv=cv, scoring=scoring)
            for m in scoring:
                all_scores[m].extend(res["test_" + m])
        row = {"experiment": exp_name, "model": model_name}
        for m in scoring:
            row[m] = round(np.mean(all_scores[m]), 2)
            row[m + "_sd"] = round(np.std(all_scores[m]), 2)
        rows.append(row)

results = pd.DataFrame(rows)
print(results.to_string(index=False))
results.to_csv("../report/results_v2_repeated.csv", index=False)

   experiment               model  accuracy  accuracy_sd  precision  precision_sd  recall  recall_sd   f1  f1_sd  roc_auc  roc_auc_sd
  B (no MMSE)            Baseline      0.55         0.01       0.00          0.00    0.00       0.00 0.00   0.00     0.50        0.00
  B (no MMSE) Logistic Regression      0.70         0.07       0.69          0.09    0.64       0.12 0.65   0.09     0.76        0.09
  B (no MMSE)       Random Forest      0.63         0.08       0.61          0.11    0.56       0.11 0.57   0.09     0.69        0.09
  B (no MMSE)             XGBoost      0.64         0.08       0.61          0.10    0.59       0.13 0.59   0.10     0.69        0.08
A (with MMSE)            Baseline      0.55         0.01       0.00          0.00    0.00       0.00 0.00   0.00     0.50        0.00
A (with MMSE) Logistic Regression      0.81         0.05       0.84          0.08    0.73       0.09 0.77   0.07     0.91        0.04
A (with MMSE)       Random Forest      0.80         0.05      